# Main script to calculate dimensionality

Calculates dimensionality for all observers and for the pooled observer O1 - O5.
It applies the signifficance criterion as in cblearn.

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sklearn

import cblearn

from load_data import load_all, DATA_ROOT
from dimensionality_estimation import estimate_dimensionality

from sdt_utils_and_logodds import recalculate_dimensionality
from io_json import save_json_dataclass

# seeds the SOE initialisations and the cross-validation splits of every fit
SEED = 0

# Conditions whose dimensionality depends on the SOE initialisation (seed_stability.py,
# data/dimensionality_seed_stability_published_inputs.csv). Rule: report the majority dimensionality over
# SOE seeds 0-9 with the cross-validation splits at SEED, and fit it with the lowest SOE
# seed that gives that majority. O2 / odd one out / greebles (2, in 8 of 10 seeds) and
# O4 / odd one out / taxonomic (1, in 9 of 10) get it with SOE seed 0, so only
# P1 / odd one out / action (2, in 8 of 10) needs another one.
SOE_SEED_OVERRIDE = {('P1', 'odd one out', 'action'): 1}

print('cblearn:', cblearn.__version__)
print('sklearn:', sklearn.__version__)

cblearn: 0.3.0
sklearn: 1.5.2


# Reading data

### reading number of images in each set

In [2]:
set_names = ('action', 'greebles', 'taxonomic')
objects = {}
for name in set_names:
    df = pd.read_csv(DATA_ROOT / f'resources/{name}_info.csv')
    df['image_set'] = name
    objects[name] = df
    print(f"{name}: {len(df)} images")

action: 18 images
greebles: 18 images
taxonomic: 18 images


### reading triplet data from all participants

In [3]:
triplet_df = load_all()

In [4]:
triplet_df

,task,observer,session,date,image_set,n_timeout,responses,raw_triplets,raw_responses,rt,triplet_index,embedding_ready_triplets,n_triplets,n_trials,mean_rt,block,file,triplet_set,shuffle
0,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,action,0,"[-1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, 1, -1, -...","[[3.0, 14.0, 16.0], [6.0, 5.0, 10.0], [1.0, 8....","[red, red, green, green, red, red, green, red,...",6 5.12 7 1.20 8 7.29 10 1.1...,"[646, 636, 629, 600, 634, 643, 644, 616, 628, ...","[[3, 16, 14], [6, 5, 10], [1, 8, 15], [2, 4, 1...",300,300,2.033267,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
1,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,greebles,1,"[1, -1, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1...","[[0.0, 7.0, 12.0], [5.0, 8.0, 10.0], [10.0, 9....","[green, red, green, red, green, green, red, gr...",2 5.63 11 2.77 13 2.37 14 2.9...,"[639, 628, 636, 646, 623, 614, 600, 617, 644, ...","[[0, 7, 12], [5, 10, 8], [10, 9, 16], [0, 7, 1...",299,299,3.290167,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
2,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,taxonomic,2,"[1, 1, 1, 1, 1, -1, 1, 1, 1, -1, -1, -1, -1, -...","[[0.0, 1.0, 3.0], [0.0, 1.0, 5.0], [0.0, 12.0,...","[green, red, green, green, green, red, green, ...",1 1.45 3 1.10 4 8.65 5 2.2...,"[645, 648, 636, 616, 608, 603, 632, 641, 643, ...","[[0, 1, 3], [0, 1, 5], [0, 12, 9], [6, 3, 9], ...",298,298,2.140067,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
3,odd one out,O2,1d,2023-02-09 16:45:57.861000+01:00,action,0,"[2, 0, 2, 2, 1, 1, 0, 0, 1, 1, 2, 0, 0, 2, 1, ...","[[15.0, 8.0, 13.0], [6.0, 7.0, 15.0], [6.0, 5....","[green, red, green, red, yellow, yellow, red, ...",2 1.60 3 1.12 4 0.96 17 0.8...,"[0, 12, 38, 39, 26, 35, 18, 10, 19, 30, 20, 47...","[[15.0, 8.0, 13.0], [7.0, 15.0, 6.0], [6.0, 5....",600,300,1.357933,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",../experiment/data/odd one out_O2_1d_object_tr...,1,d
4,odd one out,O2,1d,2023-02-09 16:45:57.861000+01:00,greebles,0,"[1, 1, 2, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 2, 0, ...","[[0.0, 8.0, 13.0], [12.0, 3.0, 14.0], [8.0, 11...","[yellow, yellow, green, green, red, yellow, gr...",1 2.61 6 3.99 7 1.34 11 1.7...,"[4, 19, 28, 27, 14, 43, 30, 33, 6, 45, 16, 0, ...","[[0.0, 13.0, 8.0], [12.0, 14.0, 3.0], [8.0, 11...",600,300,1.814967,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",../experiment/data/odd one out_O2_1d_object_tr...,1,d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274,classic,O4,2e,2023-04-25 16:07:46.491000+02:00,greebles,0,"[1, -1, 1, 1, 1, -1, 1, 1, 1, -1, -1, -1, 1, 1...","[[0.0, 1.0, 3.0], [12.0, 16.0, 15.0], [7.0, 10...","[green, green, green, red, green, green, green...",2 0.95 10 2.31 14 3.34 19 1.3...,"[324, 300, 322, 308, 338, 336, 329, 348, 334, ...","[[0, 1, 3], [12, 15, 16], [7, 10, 15], [1, 7, ...",300,300,1.587100,"[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ...",../experiment_mid_lab/data/classic_O4_2e_objec...,2,e
275,classic,O4,2e,2023-04-25 16:07:46.491000+02:00,taxonomic,0,"[-1, 1, -1, 1, -1, 1, -1, -1, 1, 1, 1, -1, -1,...","[[2.0, 16.0, 5.0], [12.0, 0.0, 16.0], [9.0, 6....","[red, red, green, green, green, green, red, gr...",1 2.39 3 2.61 4 0.68 8 0.9...,"[344, 300, 302, 310, 328, 304, 303, 348, 340, ...","[[2, 5, 16], [12, 0, 16], [9, 11, 6], [13, 4, ...",300,300,1.520833,"[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ...",../experiment_mid_lab/data/classic_O4_2e_objec...,2,e
276,classic,O4,2d,2023-02-10 19:18:23.026000+01:00,action,0,"[1, -1, -1, -1, 1, -1, 1, -1, 1, -1, 1, 1, 1, ...","[[1.0, 12.0, 17.0], [1.0, 17.0, 4.0], [1.0, 7....","[red, red, red, green, red, green, red, red, r...",2 2.67 4 1.20 5 1.52 6 1.1...,"[317, 316, 308, 341, 315, 329, 314, 309, 313, ...","[[1, 12, 17], [1, 4, 17], [1, 10, 7], [6, 14, ...",300,300,0.908833,"[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7,

Each row in the dataframe is *one block*, corresponding to about 300 trials.

In [5]:
# This bit of code is dealing with the fact, that P1 has the first repetitions and O5 repetitions 2 & 3: 
temp = triplet_df[(triplet_df['observer'] == 'P1') & (triplet_df['session'].isin(['1a', '1b', '2b', '2a']))]
temp
# Change 'observer' to 'O5' in temp
temp.loc[:,'observer'] = 'O5'

In [6]:
triplet_df = pd.concat([triplet_df, temp], ignore_index=True)
triplet_df

,task,observer,session,date,image_set,n_timeout,responses,raw_triplets,raw_responses,rt,triplet_index,embedding_ready_triplets,n_triplets,n_trials,mean_rt,block,file,triplet_set,shuffle
0,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,action,0,"[-1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, 1, -1, -...","[[3.0, 14.0, 16.0], [6.0, 5.0, 10.0], [1.0, 8....","[red, red, green, green, red, red, green, red,...",6 5.12 7 1.20 8 7.29 10 1.1...,"[646, 636, 629, 600, 634, 643, 644, 616, 628, ...","[[3, 16, 14], [6, 5, 10], [1, 8, 15], [2, 4, 1...",300,300,2.033267,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
1,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,greebles,1,"[1, -1, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1...","[[0.0, 7.0, 12.0], [5.0, 8.0, 10.0], [10.0, 9....","[green, red, green, red, green, green, red, gr...",2 5.63 11 2.77 13 2.37 14 2.9...,"[639, 628, 636, 646, 623, 614, 600, 617, 644, ...","[[0, 7, 12], [5, 10, 8], [10, 9, 16], [0, 7, 1...",299,299,3.290167,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
2,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,taxonomic,2,"[1, 1, 1, 1, 1, -1, 1, 1, 1, -1, -1, -1, -1, -...","[[0.0, 1.0, 3.0], [0.0, 1.0, 5.0], [0.0, 12.0,...","[green, red, green, green, green, red, green, ...",1 1.45 3 1.10 4 8.65 5 2.2...,"[645, 648, 636, 616, 608, 603, 632, 641, 643, ...","[[0, 1, 3], [0, 1, 5], [0, 12, 9], [6, 3, 9], ...",298,298,2.140067,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
3,odd one out,O2,1d,2023-02-09 16:45:57.861000+01:00,action,0,"[2, 0, 2, 2, 1, 1, 0, 0, 1, 1, 2, 0, 0, 2, 1, ...","[[15.0, 8.0, 13.0], [6.0, 7.0, 15.0], [6.0, 5....","[green, red, green, red, yellow, yellow, red, ...",2 1.60 3 1.12 4 0.96 17 0.8...,"[0, 12, 38, 39, 26, 35, 18, 10, 19, 30, 20, 47...","[[15.0, 8.0, 13.0], [7.0, 15.0, 6.0], [6.0, 5....",600,300,1.357933,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",../experiment/data/odd one out_O2_1d_object_tr...,1,d
4,odd one out,O2,1d,2023-02-09 16:45:57.861000+01:00,greebles,0,"[1, 1, 2, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 2, 0, ...","[[0.0, 8.0, 13.0], [12.0, 3.0, 14.0], [8.0, 11...","[yellow, yellow, green, green, red, yellow, gr...",1 2.61 6 3.99 7 1.34 11 1.7...,"[4, 19, 28, 27, 14, 43, 30, 33, 6, 45, 16, 0, ...","[[0.0, 13.0, 8.0], [12.0, 14.0, 3.0], [8.0, 11...",600,300,1.814967,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",../experiment/data/odd one out_O2_1d_object_tr...,1,d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,classic,O5,1a,2023-01-10 15:58:43.332000+01:00,greebles,6,"[1, -1, 1, -1, 1, -1, -1, 1, 1, -1, 1, 1, 1, 1...","[[10.0, 4.0, 12.0], [8.0, 3.0, 14.0], [2.0, 0....","[red, red, green, green, red, green, green, re...",1 2.47 2 2.35 4 1.98 8 1.9...,"[7, 16, 21, 37, 45, 8, 29, 24, 14, 39, 19, 38,...","[[10, 4, 12], [8, 14, 3], [2, 0, 7], [13, 14, ...",294,294,2.913946,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",../experiment/data/classic_P1_1a_object_triple...,1,a
287,classic,O5,1a,2023-01-10 15:58:43.332000+01:00,taxonomic,3,"[1, -1, 1, -1, -1, -1, 1, 1, -1, -1, -1, -1, 1...","[[4.0, 7.0, 14.0], [1.0, 14.0, 11.0], [13.0, 1...","[red, red, red, green, red, green, red, green,...",6 2.33 7 1.34 12 1.39 13 1.7...,"[22, 49, 17, 16, 26, 40, 41, 34, 18, 32, 13, 1...","[[4, 7, 14], [1, 11, 14], [13, 11, 15], [3, 17...",297,297,2.436532,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",../experiment/data/classic_P1_1a_object_triple...,1,a
288,odd one out,O5,1b,2023-01-12 09:38:12.069000+01:00,action,0,"[1, 1, 2, 0, 0, 0, 1, 2, 0, 0, 1, 2, 0, 0, 2, ...","[[15.0, 3.0, 7.0], [14.0, 11.0, 13.0], [7.0, 9...","[yellow, yellow, red, green, green, green, yel...",5 1.41 10 1.50 12 1.90 16 3.4...,"[4, 33, 16, 24, 21, 3, 25, 48, 13, 18, 42, 9, ...","[[15.0, 7.0, 3.0], [14.0, 13.0, 11.0], [7.0, 9...",600,300,2.516433,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

## Special case for odd-one-out

For the task odd-one-out two triplets can be infered from each trial. This is the default behavior when reading the data.
However this means that for the double amount of data enters the embedding estimation, something that is less comparable than
if the same amount of data would be used. For that reason we create a new 'task' in the dataframe, which selects only one of the two triplets that are deduced from each odd-one-out trial.

In [7]:
def select_one_triplet_oddoneout(arr):

    idx_half = int(arr.shape[0]/2)

    output_arr = np.zeros((idx_half, arr.shape[1]))
    
    # iterate across rows in array
    for j in range(idx_half):
        # one triplet
        t1 = arr[j]
        #print(t1)
        # the other triplet
        t2 = arr[idx_half+j]
        #print(t2)

        ## checking that array is in the correct formatting
        np.testing.assert_equal(np.unique(t1), np.unique(t2))

        if np.random.randint(0, 2)==0:
            # keep t1
            output_arr[j, :] = t1
        else:
            # keep t2
            output_arr[j, :] = t2

    return output_arr


In [8]:
data = []
np.random.seed(0)

for ix, d in triplet_df.iterrows():
    if d['task'] == 'classic':
        continue

    arr = d['embedding_ready_triplets']
    new_array = select_one_triplet_oddoneout(arr)

    #new_array
    data.append(['odd-one-out-oneoftwo', d['observer'], d['session'], d['date'], d['image_set'], new_array])

In [9]:
newdf = pd.DataFrame(data, columns=['task', 'observer', 'session', 'date', 'image_set', 'embedding_ready_triplets'])

In [10]:
newdf

,task,observer,session,date,image_set,embedding_ready_triplets
0,odd-one-out-oneoftwo,O2,1d,2023-02-09 16:45:57.861000+01:00,action,"[[15.0, 8.0, 13.0], [15.0, 7.0, 6.0], [5.0, 6...."
1,odd-one-out-oneoftwo,O2,1d,2023-02-09 16:45:57.861000+01:00,greebles,"[[13.0, 0.0, 8.0], [12.0, 14.0, 3.0], [11.0, 8..."
2,odd-one-out-oneoftwo,O2,1d,2023-02-09 16:45:57.861000+01:00,taxonomic,"[[4.0, 10.0, 12.0], [17.0, 6.0, 9.0], [13.0, 3..."
3,odd-one-out-oneoftwo,O5,1f,2023-07-15 10:58:32.557000+02:00,action,"[[7.0, 15.0, 3.0], [9.0, 15.0, 5.0], [1.0, 4.0..."
4,odd-one-out-oneoftwo,O5,1f,2023-07-15 10:58:32.557000+02:00,greebles,"[[1.0, 9.0, 12.0], [5.0, 11.0, 13.0], [16.0, 1..."
...,...,...,...,...,...,...
139,odd-one-out-oneoftwo,O5,2a,2023-01-12 10:49:50.175000+01:00,greebles,"[[16.0, 3.0, 13.0], [17.0, 3.0, 7.0], [2.0, 8...."
140,odd-one-out-oneoftwo,O5,2a,2023-01-12 10:49:50.175000+01:00,taxonomic,"[[4.0, 13.0, 14.0], [15.0, 9.0, 6.0], [4.0, 3...."
141,odd-one-out-oneoftwo,O5,1b,2023-01-12 09:38:12.069000+01:00,action,"[[7.0, 15.0, 3.0], [13.0, 14.0, 11.0], [9.0, 7..."
142,odd-one-out-oneoftwo,O5,1b,2023-01-12 09:38:12.069000+01:00,greebles,"[[14.0, 8.0, 3.0], [1.0, 12.0, 9.0], [15.0, 9...."


In [11]:
triplet_df = pd.concat((triplet_df, newdf))

In [12]:
triplet_df

,task,observer,session,date,image_set,n_timeout,responses,raw_triplets,raw_responses,rt,triplet_index,embedding_ready_triplets,n_triplets,n_trials,mean_rt,block,file,triplet_set,shuffle
0,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,action,0.0,"[-1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, 1, -1, -...","[[3.0, 14.0, 16.0], [6.0, 5.0, 10.0], [1.0, 8....","[red, red, green, green, red, red, green, red,...",6 5.12 7 1.20 8 7.29 10 1.1...,"[646, 636, 629, 600, 634, 643, 644, 616, 628, ...","[[3, 16, 14], [6, 5, 10], [1, 8, 15], [2, 4, 1...",300.0,300.0,2.033267,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
1,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,greebles,1.0,"[1, -1, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1...","[[0.0, 7.0, 12.0], [5.0, 8.0, 10.0], [10.0, 9....","[green, red, green, red, green, green, red, gr...",2 5.63 11 2.77 13 2.37 14 2.9...,"[639, 628, 636, 646, 623, 614, 600, 617, 644, ...","[[0, 7, 12], [5, 10, 8], [10, 9, 16], [0, 7, 1...",299.0,299.0,3.290167,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
2,classic,P2,3a,2023-12-04 16:57:06.734000+01:00,taxonomic,2.0,"[1, 1, 1, 1, 1, -1, 1, 1, 1, -1, -1, -1, -1, -...","[[0.0, 1.0, 3.0], [0.0, 1.0, 5.0], [0.0, 12.0,...","[green, red, green, green, green, red, green, ...",1 1.45 3 1.10 4 8.65 5 2.2...,"[645, 648, 636, 616, 608, 603, 632, 641, 643, ...","[[0, 1, 3], [0, 1, 5], [0, 12, 9], [6, 3, 9], ...",298.0,298.0,2.140067,"[13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 1...",../experiment/data/classic_P2_3a_object_triple...,3,a
3,odd one out,O2,1d,2023-02-09 16:45:57.861000+01:00,action,0.0,"[2, 0, 2, 2, 1, 1, 0, 0, 1, 1, 2, 0, 0, 2, 1, ...","[[15.0, 8.0, 13.0], [6.0, 7.0, 15.0], [6.0, 5....","[green, red, green, red, yellow, yellow, red, ...",2 1.60 3 1.12 4 0.96 17 0.8...,"[0, 12, 38, 39, 26, 35, 18, 10, 19, 30, 20, 47...","[[15.0, 8.0, 13.0], [7.0, 15.0, 6.0], [6.0, 5....",600.0,300.0,1.357933,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",../experiment/data/odd one out_O2_1d_object_tr...,1,d
4,odd one out,O2,1d,2023-02-09 16:45:57.861000+01:00,greebles,0.0,"[1, 1, 2, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 2, 0, ...","[[0.0, 8.0, 13.0], [12.0, 3.0, 14.0], [8.0, 11...","[yellow, yellow, green, green, red, yellow, gr...",1 2.61 6 3.99 7 1.34 11 1.7...,"[4, 19, 28, 27, 14, 43, 30, 33, 6, 45, 16, 0, ...","[[0.0, 13.0, 8.0], [12.0, 14.0, 3.0], [8.0, 11...",600.0,300.0,1.814967,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",../experiment/data/odd one out_O2_1d_object_tr...,1,d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,odd-one-out-oneoftwo,O5,2a,2023-01-12 10:49:50.175000+01:00,greebles,NaN,NaN,NaN,NaN,NaN,NaN,"[[16.0, 3.0, 13.0], [17.0, 3.0, 7.0], [2.0, 8....",NaN,NaN,NaN,NaN,NaN,NaN,NaN
140,odd-one-out-oneoftwo,O5,2a,2023-01-12 10:49:50.175000+01:00,taxonomic,NaN,NaN,NaN,NaN,NaN,NaN,"[[4.0, 13.0, 14.0], [15.0, 9.0, 6.0], [4.0, 3....",NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,odd-one-out-oneoftwo,O5,1b,2023-01-12 09:38:12.069000+01:00,action,NaN,NaN,NaN,NaN,NaN,NaN,"[[7.0, 15.0, 3.0], [13.0, 14.0, 11.0], [9.0, 7...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,odd-one-out-oneoftwo,O5,1b,2023-01-12 09:38:12.069000+01:00,greebles,NaN,NaN,NaN,NaN,NaN,NaN,"[[14.0, 8.0, 3.0], [1.0, 12.0, 9.0], [15.0, 9....",NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Fitting embeddings to aggregates of data

Here we put different sessions together and then run ordinal embedding on it, finding the dimensionality.
Following Künstle et al. (2022)

In [13]:
observers = triplet_df['observer'].unique()
observers

array(['P2', 'O2', 'O1', 'O5', 'P1', 'O4', 'O3'], dtype=object)

In [14]:
# we join all sessions for an observer on a particular task and image set
joined_df = triplet_df.groupby(['observer', 'task', 'image_set'])['embedding_ready_triplets'].apply(np.vstack).reset_index()
joined_df

,observer,task,image_set,embedding_ready_triplets
0,O1,classic,action,"[[3, 7, 15], [2, 3, 6], [9, 3, 11], [6, 1, 14]..."
1,O1,classic,greebles,"[[12, 17, 9], [2, 3, 7], [9, 10, 0], [6, 10, 1..."
2,O1,classic,taxonomic,"[[2, 12, 4], [1, 11, 10], [13, 11, 15], [5, 16..."
3,O1,odd one out,action,"[[17.0, 4.0, 11.0], [2.0, 4.0, 14.0], [17.0, 1..."
4,O1,odd one out,greebles,"[[4.0, 16.0, 7.0], [13.0, 8.0, 3.0], [16.0, 13..."
...,...,...,...,...
58,P2,odd one out,greebles,"[[0.0, 1.0, 3.0], [0.0, 16.0, 12.0], [8.0, 11...."
59,P2,odd one out,taxonomic,"[[7.0, 4.0, 5.0], [7.0, 15.0, 5.0], [1.0, 0.0,..."
60,P2,odd-one-out-oneoftwo,action,"[[3.0, 4.0, 14.0], [14.0, 17.0, 9.0], [4.0, 3...."
61,P2,odd-one-out-oneoftwo,greebles,"[[1.0, 0.0, 3.0], [0.0, 16.0, 12.0], [11.0, 8...."


In [15]:
raw_triplets = triplet_df.groupby(['observer', 'task', 'image_set'])['raw_triplets'].apply(np.vstack).reset_index()
raw_triplets

,observer,task,image_set,raw_triplets
0,O1,classic,action,"[[3.0, 15.0, 7.0], [2.0, 3.0, 6.0], [9.0, 3.0,..."
1,O1,classic,greebles,"[[12.0, 9.0, 17.0], [2.0, 3.0, 7.0], [9.0, 0.0..."
2,O1,classic,taxonomic,"[[2.0, 4.0, 12.0], [1.0, 11.0, 10.0], [13.0, 1..."
3,O1,odd one out,action,"[[17.0, 4.0, 11.0], [2.0, 4.0, 14.0], [17.0, 8..."
4,O1,odd one out,greebles,"[[4.0, 7.0, 16.0], [13.0, 3.0, 8.0], [16.0, 3...."
...,...,...,...,...
58,P2,odd one out,greebles,"[[0.0, 1.0, 3.0], [12.0, 0.0, 16.0], [8.0, 11...."
59,P2,odd one out,taxonomic,"[[7.0, 4.0, 5.0], [7.0, 5.0, 15.0], [1.0, 0.0,..."
60,P2,odd-one-out-oneoftwo,action,"[[nan], [nan], [nan], [nan], [nan], [nan], [na..."
61,P2,odd-one-out-oneoftwo,greebles,"[[nan], [nan], [nan], [nan], [nan], [nan], [na..."


In [16]:
responses = triplet_df.groupby(['observer', 'task', 'image_set'])['responses'].apply(np.hstack).reset_index()
responses

,observer,task,image_set,responses
0,O1,classic,action,"[-1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, 1, -1, -1..."
1,O1,classic,greebles,"[-1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, -1, -1..."
2,O1,classic,taxonomic,"[-1, 1, 1, 1, 1, 1, 1, -1, -1, 1, -1, -1, -1, ..."
3,O1,odd one out,action,"[2, 2, 1, 2, 1, 1, 2, 2, 2, 2, 1, 0, 1, 2, 2, ..."
4,O1,odd one out,greebles,"[1, 1, 1, 0, 0, 0, 0, 2, 1, 0, 0, 1, 1, 1, 1, ..."
...,...,...,...,...
58,P2,odd one out,greebles,"[2, 0, 2, 2, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 2, ..."
59,P2,odd one out,taxonomic,"[2, 1, 2, 0, 1, 0, 1, 0, 1, 2, 1, 1, 0, 1, 0, ..."
60,P2,odd-one-out-oneoftwo,action,"[nan, nan, nan, nan, nan, nan, nan, nan, nan]"
61,P2,odd-one-out-oneoftwo,greebles,"[nan, nan, nan, nan, nan, nan, nan, nan, nan]"


In [17]:
joined_df = joined_df.merge(raw_triplets, on=['observer', 'task', 'image_set'])
joined_df = joined_df.merge(responses, on=['observer', 'task', 'image_set'])
joined_df

,observer,task,image_set,embedding_ready_triplets,raw_triplets,responses
0,O1,classic,action,"[[3, 7, 15], [2, 3, 6], [9, 3, 11], [6, 1, 14]...","[[3.0, 15.0, 7.0], [2.0, 3.0, 6.0], [9.0, 3.0,...","[-1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, 1, -1, -1..."
1,O1,classic,greebles,"[[12, 17, 9], [2, 3, 7], [9, 10, 0], [6, 10, 1...","[[12.0, 9.0, 17.0], [2.0, 3.0, 7.0], [9.0, 0.0...","[-1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, -1, -1..."
2,O1,classic,taxonomic,"[[2, 12, 4], [1, 11, 10], [13, 11, 15], [5, 16...","[[2.0, 4.0, 12.0], [1.0, 11.0, 10.0], [13.0, 1...","[-1, 1, 1, 1, 1, 1, 1, -1, -1, 1, -1, -1, -1, ..."
3,O1,odd one out,action,"[[17.0, 4.0, 11.0], [2.0, 4.0, 14.0], [17.0, 1...","[[17.0, 4.0, 11.0], [2.0, 4.0, 14.0], [17.0, 8...","[2, 2, 1, 2, 1, 1, 2, 2, 2, 2, 1, 0, 1, 2, 2, ..."
4,O1,odd one out,greebles,"[[4.0, 16.0, 7.0], [13.0, 8.0, 3.0], [16.0, 13...","[[4.0, 7.0, 16.0], [13.0, 3.0, 8.0], [16.0, 3....","[1, 1, 1, 0, 0, 0, 0, 2, 1, 0, 0, 1, 1, 1, 1, ..."
...,...,...,...,...,...,...
58,P2,odd one out,greebles,"[[0.0, 1.0, 3.0], [0.0, 16.0, 12.0], [8.0, 11....","[[0.0, 1.0, 3.0], [12.0, 0.0, 16.0], [8.0, 11....","[2, 0, 2, 2, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 2, ..."
59,P2,odd one out,taxonomic,"[[7.0, 4.0, 5.0], [7.0, 15.0, 5.0], [1.0, 0.0,...","[[7.0, 4.0, 5.0], [7.0, 5.0, 15.0], [1.0, 0.0,...","[2, 1, 2, 0, 1, 0, 1, 0, 1, 2, 1, 1, 0, 1, 0, ..."
60,P2,odd-one-out-oneoftwo,action,"[[3.0, 4.0, 14.0], [14.0, 17.0, 9.0], [4.0, 3....","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan]"
61,P2,odd-one-out-oneoftwo,greebles,"[[1.0, 0.0, 3.0], [0.0, 16.0, 12.0], [11.0, 8....","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan]"


In [18]:
Nts = []
for ix, d in joined_df.iterrows():
    
    obs = d['observer']
    image_set = d['image_set']
    task = d['task']
    
    triplets = d['embedding_ready_triplets']
    Nt = len(triplets)
    Nts.append(Nt)
    print(f"{obs} - {task} - {image_set} - {Nt} triplets")
    
joined_df['Nt'] = Nts

O1 - classic - action - 1800 triplets
O1 - classic - greebles - 1799 triplets
O1 - classic - taxonomic - 1800 triplets
O1 - odd one out - action - 3600 triplets
O1 - odd one out - greebles - 3600 triplets
O1 - odd one out - taxonomic - 3598 triplets
O1 - odd-one-out-oneoftwo - action - 1800 triplets
O1 - odd-one-out-oneoftwo - greebles - 1800 triplets
O1 - odd-one-out-oneoftwo - taxonomic - 1799 triplets
O2 - classic - action - 1799 triplets
O2 - classic - greebles - 1800 triplets
O2 - classic - taxonomic - 1800 triplets
O2 - odd one out - action - 3600 triplets
O2 - odd one out - greebles - 3600 triplets
O2 - odd one out - taxonomic - 3600 triplets
O2 - odd-one-out-oneoftwo - action - 1800 triplets
O2 - odd-one-out-oneoftwo - greebles - 1800 triplets
O2 - odd-one-out-oneoftwo - taxonomic - 1800 triplets
O3 - classic - action - 1800 triplets
O3 - classic - greebles - 1800 triplets
O3 - classic - taxonomic - 1800 triplets
O3 - odd one out - action - 3600 triplets
O3 - odd one out - gree

In [19]:
def estimate_dimensionality_rows_dataframe(df):

    dimensionality = []
    embedding = []
    
    Nrows = len(df)
    
    print('estimating dimensionality...')
    
    for ix, d in df.iterrows():
            
        obs = d['observer']
        image_set = d['image_set']
        task = d['task']
        triplets = d['embedding_ready_triplets']
        Nt = len(triplets)
        
        # dimensions 1 to 5, 5 splits x 30 repeats, refit at the estimated dimension;
        # the cross-validation splits are seeded with SEED, SOE with SEED unless overridden
        soe_seed = SOE_SEED_OVERRIDE.get((obs, task, image_set), SEED)
        dim_result, fitted_embedding = estimate_dimensionality(triplets, random_state=SEED,
                                                               soe_random_state=soe_seed)
    
        # saving triplet accuracy and statistics
        save_json_dataclass(dim_result, f"../data/triplet_accuracy_vs_dimensionality-{obs}-{task}-{image_set}.json")
    
        dimensionality.append(dim_result.estimated_dimension)
        embedding.append(fitted_embedding.tolist())
    
        assert(fitted_embedding.shape[1]==dim_result.estimated_dimension)
        
        print(f"{obs} - {task} - {image_set} - {Nt} triplets: {dim_result.estimated_dimension} dimension(s)")
    
    df['dimensionality'] = dimensionality
    df['embedding'] = embedding

    return df

In [20]:
joined_df = estimate_dimensionality_rows_dataframe(joined_df)

estimating dimensionality...


O1 - classic - action - 1800 triplets: 2 dimension(s)


O1 - classic - greebles - 1799 triplets: 2 dimension(s)


O1 - classic - taxonomic - 1800 triplets: 3 dimension(s)


O1 - odd one out - action - 3600 triplets: 2 dimension(s)


...

P2 - odd one out - greebles - 4848 triplets: 1 dimension(s)


P2 - odd one out - taxonomic - 4880 triplets: 3 dimension(s)


P2 - odd-one-out-oneoftwo - action - 2419 triplets: 1 dimension(s)


P2 - odd-one-out-oneoftwo - greebles - 2424 triplets: 1 dimension(s)


P2 - odd-one-out-oneoftwo - taxonomic - 2440 triplets: 3 dimension(s)


In [21]:
joined_df['triplets'] = joined_df['embedding_ready_triplets'].apply(lambda x: x.tolist())
joined_df['raw_triplets'] = joined_df['raw_triplets'].apply(lambda x: x.tolist())
joined_df['responses'] = joined_df['responses'].apply(lambda x: x.tolist())

In [22]:
joined_df.to_csv(f"../data/found-dimensionality-embeddings-aggregated-per-observer-nsplit-5-nrepeat-30.csv", index=False)

In [23]:
joined_df

,observer,task,image_set,embedding_ready_triplets,raw_triplets,responses,Nt,dimensionality,embedding,triplets
0,O1,classic,action,"[[3, 7, 15], [2, 3, 6], [9, 3, 11], [6, 1, 14]...","[[3.0, 15.0, 7.0], [2.0, 3.0, 6.0], [9.0, 3.0,...","[-1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, 1, -1, -1...",1800,2,"[[-0.3028389954218184, -0.2241804285211048], [...","[[3, 7, 15], [2, 3, 6], [9, 3, 11], [6, 1, 14]..."
1,O1,classic,greebles,"[[12, 17, 9], [2, 3, 7], [9, 10, 0], [6, 10, 1...","[[12.0, 9.0, 17.0], [2.0, 3.0, 7.0], [9.0, 0.0...","[-1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, -1, -1...",1799,2,"[[-0.3172726481560712, 0.13491133855648693], [...","[[12, 17, 9], [2, 3, 7], [9, 10, 0], [6, 10, 1..."
2,O1,classic,taxonomic,"[[2, 12, 4], [1, 11, 10], [13, 11, 15], [5, 16...","[[2.0, 4.0, 12.0], [1.0, 11.0, 10.0], [13.0, 1...","[-1, 1, 1, 1, 1, 1, 1, -1, -1, 1, -1, -1, -1, ...",1800,3,"[[0.3147524635139672, -0.3093693311665411, -0....","[[2, 12, 4], [1, 11, 10], [13, 11, 15], [5, 16..."
3,O1,odd one out,action,"[[17.0, 4.0, 11.0], [2.0, 4.0, 14.0], [17.0, 1...","[[17.0, 4.0, 11.0], [2.0, 4.0, 14.0], [17.0, 8...","[2, 2, 1, 2, 1, 1, 2, 2, 2, 2, 1, 0, 1, 2, 2, ...",3600,2,"[[0.17558052905052524, 0.11925394380841339], [...","[[17.0, 4.0, 11.0], [2.0, 4.0, 14.0], [17.0, 1..."
4,O1,odd one out,greebles,"[[4.0, 16.0, 7.0], [13.0, 8.0, 3.0], [16.0, 13...","[[4.0, 7.0, 16.0], [13.0, 3.0, 8.0], [16.0, 3....","[1, 1, 1, 0, 0, 0, 0, 2, 1, 0, 0, 1, 1, 1, 1, ...",3600,1,"[[-0.29699831714443525], [-0.29699749922456087...","[[4.0, 16.0, 7.0], [13.0, 8.0, 3.0], [16.0, 13..."
...,...,...,...,...,...,...,...,...,...,...
58,P2,odd one out,greebles,"[[0.0, 1.0, 3.0], [0.0, 16.0, 12.0], [8.0, 11....","[[0.0, 1.0, 3.0], [12.0, 0.0, 16.0], [8.0, 11....","[2, 0, 2, 2, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 2, ...",4848,1,"[[0.06627244416414235], [0.06627281755590231],...","[[0.0, 1.0, 3.0], [0.0, 16.0, 12.0], [8.0, 11...."
59,P2,odd one out,taxonomic,"[[7.0, 4.0, 5.0], [7.0, 15.0, 5.0], [1.0, 0.0,...","[[7.0, 4.0, 5.0], [7.0, 5.0, 15.0], [1.0, 0.0,...","[2, 1, 2, 0, 1, 0, 1, 0, 1, 2, 1, 1, 0, 1, 0, ...",4880,3,"[[0.05288306391455245, -0.045209839772125046, ...","[[7.0, 4.0, 5.0], [7.0, 15.0, 5.0], [1.0, 0.0,..."
60,P2,odd-one-out-oneoftwo,action,"[[3.0, 4.0, 14.0], [14.0, 17.0, 9.0], [4.0, 3....","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan]",2419,1,"[[-0.22561512869670078], [-0.22072239527855916...","[[3.0, 4.0, 14.0], [14.0, 17.0, 9.0], [4.0, 3...."
61,P2,odd-one-out-oneoftwo,greebles,"[[1.0, 0.0, 3.0], [0.0, 16.0, 12.0], [11.0, 8....","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan]",2424,1,"[[-0.07928512870748511], [-0.07980137857120373...","[[1.0, 0.0, 3.0], [0.0, 16.0, 12.0], [11.0, 8...."


# Fitting embeddings to aggregates of data, now all observers together

In [24]:
# we now filter and only take O1 to O5
triplet_df = triplet_df[(triplet_df['observer']=='O1') | (triplet_df['observer']=='O2') | 
                        (triplet_df['observer']=='O3') | (triplet_df['observer']=='O4') | 
                        (triplet_df['observer']=='O5')]

In [25]:
joined_df_all = triplet_df.groupby(['task', 'image_set'])['embedding_ready_triplets'].apply(np.vstack).reset_index()
raw_triplets = triplet_df.groupby(['task', 'image_set'])['raw_triplets'].apply(np.vstack).reset_index()
responses = triplet_df.groupby(['task', 'image_set'])['responses'].apply(np.hstack).reset_index()

joined_df_all = joined_df_all.merge(raw_triplets, on=['task', 'image_set'])
joined_df_all = joined_df_all.merge(responses, on=['task', 'image_set'])
joined_df_all['observer']= 'pooled O1 - O5'
joined_df_all



,task,image_set,embedding_ready_triplets,raw_triplets,responses,observer
0,classic,action,"[[3, 7, 15], [2, 3, 6], [9, 3, 11], [6, 1, 14]...","[[3.0, 15.0, 7.0], [2.0, 3.0, 6.0], [9.0, 3.0,...","[-1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, 1, -1, -1...",pooled O1 - O5
1,classic,greebles,"[[12, 17, 9], [2, 3, 7], [9, 10, 0], [6, 10, 1...","[[12.0, 9.0, 17.0], [2.0, 3.0, 7.0], [9.0, 0.0...","[-1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, -1, -1...",pooled O1 - O5
2,classic,taxonomic,"[[2, 12, 4], [1, 11, 10], [13, 11, 15], [5, 16...","[[2.0, 4.0, 12.0], [1.0, 11.0, 10.0], [13.0, 1...","[-1, 1, 1, 1, 1, 1, 1, -1, -1, 1, -1, -1, -1, ...",pooled O1 - O5
3,odd one out,action,"[[15.0, 8.0, 13.0], [7.0, 15.0, 6.0], [6.0, 5....","[[15.0, 8.0, 13.0], [6.0, 7.0, 15.0], [6.0, 5....","[2, 0, 2, 2, 1, 1, 0, 0, 1, 1, 2, 0, 0, 2, 1, ...",pooled O1 - O5
4,odd one out,greebles,"[[0.0, 13.0, 8.0], [12.0, 14.0, 3.0], [8.0, 11...","[[0.0, 8.0, 13.0], [12.0, 3.0, 14.0], [8.0, 11...","[1, 1, 2, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 2, 0, ...",pooled O1 - O5
5,odd one out,taxonomic,"[[4.0, 10.0, 12.0], [17.0, 6.0, 9.0], [3.0, 13...","[[12.0, 4.0, 10.0], [17.0, 6.0, 9.0], [3.0, 13...","[0, 2, 2, 1, 2, 2, 2, 0, 2, 1, 1, 0, 0, 1, 1, ...",pooled O1 - O5
6,odd-one-out-oneoftwo,action,"[[15.0, 8.0, 13.0], [15.0, 7.0, 6.0], [5.0, 6....","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",pooled O1 - O5
7,odd-one-out-oneoftwo,greebles,"[[13.0, 0.0, 8.0], [12.0, 14.0, 3.0], [11.0, 8...","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",pooled O1 - O5
8,odd-one-out-oneoftwo,taxonomic,"[[4.0, 10.0, 12.0], [17.0, 6.0, 9.0], [13.0, 3...","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",pooled O1 - O5


In [26]:
len(joined_df_all.iloc[0]['embedding_ready_triplets'])

8846

In [27]:
joined_df_all = estimate_dimensionality_rows_dataframe(joined_df_all)

estimating dimensionality...


pooled O1 - O5 - classic - action - 8846 triplets: 4 dimension(s)


pooled O1 - O5 - classic - greebles - 8839 triplets: 2 dimension(s)


pooled O1 - O5 - classic - taxonomic - 8842 triplets: 2 dimension(s)


pooled O1 - O5 - odd one out - action - 17994 triplets: 3 dimension(s)


pooled O1 - O5 - odd one out - greebles - 17996 triplets: 2 dimension(s)


pooled O1 - O5 - odd one out - taxonomic - 17994 triplets: 2 dimension(s)


pooled O1 - O5 - odd-one-out-oneoftwo - action - 8997 triplets: 1 dimension(s)


pooled O1 - O5 - odd-one-out-oneoftwo - greebles - 8998 triplets: 3 dimension(s)


pooled O1 - O5 - odd-one-out-oneoftwo - taxonomic - 8997 triplets: 2 dimension(s)


In [28]:
joined_df_all['triplets'] = joined_df_all['embedding_ready_triplets'].apply(lambda x: x.tolist())
joined_df_all['raw_triplets'] = joined_df_all['raw_triplets'].apply(lambda x: x.tolist())
joined_df_all['responses'] = joined_df_all['responses'].apply(lambda x: x.tolist())

In [29]:
joined_df_all.to_csv(f"../data/found-dimensionality-embeddings-pooled-observers-nsplit-5-nrepeat-30.csv", index=False)

In [30]:
joined_df_all

,task,image_set,embedding_ready_triplets,raw_triplets,responses,observer,dimensionality,embedding,triplets
0,classic,action,"[[3, 7, 15], [2, 3, 6], [9, 3, 11], [6, 1, 14]...","[[3.0, 15.0, 7.0], [2.0, 3.0, 6.0], [9.0, 3.0,...","[-1, 1, 1, 1, 1, 1, -1, 1, -1, 1, 1, 1, -1, -1...",pooled O1 - O5,4,"[[0.07266329078165396, -0.3445315203960514, 0....","[[3, 7, 15], [2, 3, 6], [9, 3, 11], [6, 1, 14]..."
1,classic,greebles,"[[12, 17, 9], [2, 3, 7], [9, 10, 0], [6, 10, 1...","[[12.0, 9.0, 17.0], [2.0, 3.0, 7.0], [9.0, 0.0...","[-1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1, -1, -1...",pooled O1 - O5,2,"[[0.054080578787659456, 0.27768167905882873], ...","[[12, 17, 9], [2, 3, 7], [9, 10, 0], [6, 10, 1..."
2,classic,taxonomic,"[[2, 12, 4], [1, 11, 10], [13, 11, 15], [5, 16...","[[2.0, 4.0, 12.0], [1.0, 11.0, 10.0], [13.0, 1...","[-1, 1, 1, 1, 1, 1, 1, -1, -1, 1, -1, -1, -1, ...",pooled O1 - O5,2,"[[0.5522614104883116, 0.17867273452170793], [0...","[[2, 12, 4], [1, 11, 10], [13, 11, 15], [5, 16..."
3,odd one out,action,"[[15.0, 8.0, 13.0], [7.0, 15.0, 6.0], [6.0, 5....","[[15.0, 8.0, 13.0], [6.0, 7.0, 15.0], [6.0, 5....","[2, 0, 2, 2, 1, 1, 0, 0, 1, 1, 2, 0, 0, 2, 1, ...",pooled O1 - O5,3,"[[0.1416815091483734, -0.13066201729320517, -0...","[[15.0, 8.0, 13.0], [7.0, 15.0, 6.0], [6.0, 5...."
4,odd one out,greebles,"[[0.0, 13.0, 8.0], [12.0, 14.0, 3.0], [8.0, 11...","[[0.0, 8.0, 13.0], [12.0, 3.0, 14.0], [8.0, 11...","[1, 1, 2, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 2, 0, ...",pooled O1 - O5,2,"[[-0.44577296534212807, -0.0025946406469132075...","[[0.0, 13.0, 8.0], [12.0, 14.0, 3.0], [8.0, 11..."
5,odd one out,taxonomic,"[[4.0, 10.0, 12.0], [17.0, 6.0, 9.0], [3.0, 13...","[[12.0, 4.0, 10.0], [17.0, 6.0, 9.0], [3.0, 13...","[0, 2, 2, 1, 2, 2, 2, 0, 2, 1, 1, 0, 0, 1, 1, ...",pooled O1 - O5,2,"[[-0.13427402449064277, 0.11084256168064961], ...","[[4.0, 10.0, 12.0], [17.0, 6.0, 9.0], [3.0, 13..."
6,odd-one-out-oneoftwo,action,"[[15.0, 8.0, 13.0], [15.0, 7.0, 6.0], [5.0, 6....","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",pooled O1 - O5,1,"[[-0.17356767349932384], [-0.1739989434500061]...","[[15.0, 8.0, 13.0], [15.0, 7.0, 6.0], [5.0, 6...."
7,odd-one-out-oneoftwo,greebles,"[[13.0, 0.0, 8.0], [12.0, 14.0, 3.0], [11.0, 8...","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",pooled O1 - O5,3,"[[0.08754028176092998, -0.05364487740467283, -...","[[13.0, 0.0, 8.0], [12.0, 14.0, 3.0], [11.0, 8..."
8,odd-one-out-oneoftwo,taxonomic,"[[4.0, 10.0, 12.0], [17.0, 6.0, 9.0], [13.0, 3...","[[nan], [nan], [nan], [nan], [nan], [nan], [na...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",pooled O1 - O5,2,"[[0.17776075635705696, 0.24286609794154268], [...","[[4.0, 10.0, 12.0], [17.0, 6.0, 9.0], [13.0, 3..."
